In [1]:
import json
from itertools import islice
from pathlib import Path

import pandas as pd
from datasets import load_dataset
from huggingface_hub import HfFileSystem

import io
import re
import urllib.request

from config import SEED, DATA_DIR, CATEGORIES, REVIEWS_PER_SUMMARY, MAX_WORDS_PER_REVIEW, SAMPLE_SIZE, SHUFFLE_BUFFER, DATASET_URL, REVIEW_COLUMNS, TOP_N_TOTAL

DATA_DIR.mkdir(exist_ok=True)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Defines functions that stream a review sample from Hugging Face, save it to data and load it back
def stream_reviews(category, n):
    data_file = f"{DATASET_URL}/review_categories/{category}.jsonl"

    # Streams the data from Hugging Face
    stream = load_dataset("json", data_files=data_file, split="train", streaming=True)

    rows = stream.shuffle(buffer_size=SHUFFLE_BUFFER, seed=SEED).take(n)
    df = pd.DataFrame(list(rows))[REVIEW_COLUMNS]
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
    df["category"] = category

    return df

def build_path(category, n=SAMPLE_SIZE):
    """
        Builds the path from the category
    """
    return DATA_DIR / f"reviews_{category}_{n}.parquet"

def save_review_data(category, update=False, n=SAMPLE_SIZE):
    """
        Streams a sample and saves it to a .parquet file, unless it's already saved.
    """
    path = build_path(category, n)

    if path.exists() and not update:
        print(f"{category}: already saved at {path}")
        return

    df = stream_reviews(category, n)
    df.to_parquet(path, index=False)
    print(f"{category}: saved to {path}")

def load_reviews(category, n=SAMPLE_SIZE):
    """
        Loads a saved sample from data/.
    """
    path = build_path(category, n)

    if not path.exists():
        raise FileNotFoundError(f"{path} not found.")

    return pd.read_parquet(path)

In [3]:
# Saves each categorys review sample and combines them into one reviews table
category_reviews = []
for c in CATEGORIES:
    save_review_data(c, update=True)
    category_reviews.append(load_reviews(c))

reviews = pd.concat(category_reviews, ignore_index=True)
print(f"\nTotal: {len(reviews):,} reviews")

Electronics: saved to data/reviews_Electronics_20000.parquet
Home_and_Kitchen: saved to data/reviews_Home_and_Kitchen_20000.parquet

Total: 40,000 reviews


In [4]:
# Shows the first few reviews
reviews.head()

,rating,title,text,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,category
0,5.0,Good for the application,Works very well. Plugging on either end is ni...,B001DK1ZVO,B00WTZGRRE,AGK3NYKU3CARH7ECUIKNPLUMO4OQ,2014-09-28 01:01:34.000,0,False,Electronics
1,5.0,Stand,Does just what it says,B08RYZ219J,B08RYZ219J,AFQKWXGOZ6N33TEWPESQEA7NQWIA,2021-12-22 03:30:12.537,0,True,Electronics
2,5.0,Maeline Micro USB charging cable,"Best cables I've ever had fast charging, stron...",B07Q6HT22Q,B07Q3BBNS8,AFSNGBROR4NUCU6UQCJBB4VEJOPA,2022-06-21 18:22:09.417,0,True,Electronics
3,5.0,Easy,Works great. I had to reapply it a second time...,B015JX3H48,B015JX3H48,AH7P2ZJ7NDT2NNVS7ZEGNEPT2J2Q,2016-01-28 17:06:50.000,1,True,Electronics
4,5.0,Nice and Sleek and it works,This is working great for my work MacBook to c...,B07QMMW1SC,B0B28GXPRR,AGLQQCIS5V6EGUCS5SSNHWZHQM6Q,2019-12-21 05:07:25.105,0,False,Electronics


In [5]:
# Counts how many reviews have each star rating in each category
reviews.groupby("category")["rating"].value_counts().unstack().astype(int)

rating,1.0,2.0,3.0,4.0,5.0
category,,,,,
Electronics,1505,755,1335,2891,13514
Home_and_Kitchen,1060,637,1458,3331,13514


In [6]:
# Counts verified and unverified purchases for each star rating
reviews.groupby("rating")["verified_purchase"].value_counts().unstack().astype(int)

verified_purchase,False,True
rating,,
1.0,342,2223
2.0,338,1054
3.0,851,1942
4.0,2877,3345
5.0,7434,19594


In [7]:
# Shows how many distinct products are in the sample and how many reviews each one has
per_product = reviews.groupby(["category", "parent_asin"]).size()
per_product.groupby("category").describe()[["count", "mean", "max"]]

,count,mean,max
category,,,
Electronics,15921.0,1.256202,62.0
Home_and_Kitchen,17813.0,1.122776,11.0


In [8]:
# Previews the first five products in the first category's metadata file to see what fields it has
meta_path = f"{DATASET_URL}/meta_categories/meta_{CATEGORIES[0]}.jsonl".removeprefix("hf://")

peek_rows = []
with HfFileSystem().open(meta_path) as f:
    for line in islice(f, 5):
        peek_rows.append(json.loads(line))

meta_peek = pd.DataFrame(peek_rows)

meta_peek[["parent_asin", "title", "main_category", "categories", "average_rating", "rating_number"]]

,parent_asin,title,main_category,categories,average_rating,rating_number
0,B00MCW7G9M,FS-1051 FATSHARK TELEPORTER V3 HEADSET,All Electronics,"[Electronics, Television & Video, Video Glasses]",3.5,6
1,B00YT6XQSE,Ce-H22B12-S1 4Kx2K Hdmi 4Port,All Electronics,"[Electronics, Television & Video, Accessories,...",5.0,1
2,B07SM135LS,Digi-Tatoo Decal Skin Compatible With MacBook ...,Computers,"[Electronics, Computers & Accessories, Laptop ...",4.5,246
3,B089CNGZCW,NotoCity Compatible with Vivoactive 4 band 22m...,AMAZON FASHION,"[Electronics, Wearable Technology, Clips, Arm ...",4.5,233
4,B004E2Z88O,Motorola Droid X Essentials Combo Pack,Cell Phones & Accessories,"[Electronics, Computers & Accessories, Compute...",3.8,64


In [9]:
# Defines functions that look up each products name, brand, category, average stars and review coun
HF_URL = "https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw"
META_COLUMNS = ["parent_asin", "title", "store", "main_category", "categories", "average_rating", "rating_number"]
PARENT_ASIN_PATTERN = re.compile(rb'"parent_asin": "([^"]+)"')


def pick_fields(item, columns):
    row = {}
    for key in columns:
        row[key] = item.get(key)
    return row

def scan_metadata(category, wanted_asins):
    rows = []
    url = f"{HF_URL}/meta_categories/meta_{category}.jsonl"
    with urllib.request.urlopen(url) as response:
        for line_number, line in enumerate(io.BufferedReader(response, buffer_size=16 * 2**20), start=1):
            match = PARENT_ASIN_PATTERN.search(line)
            if match and match.group(1).decode() in wanted_asins:
                item = json.loads(line)
                rows.append(pick_fields(item, META_COLUMNS))
                if len(rows) == len(wanted_asins):
                    break
            if line_number % 1_000_000 == 0:
                print(f"  {line_number:,} products scanned, {len(rows):,} matched")
    return pd.DataFrame(rows, columns=META_COLUMNS)

def category_list(path):
    if path is None:
        return []

    return list(path)

def subcategory(path):
    if len(path) > 1:
        return path[1]

    return None

def product_type(path):
    if len(path) > 0:
        return path[-1]

    return None

def build_products(meta):
    paths = meta["categories"].map(category_list)

    products = pd.DataFrame({
        "parent_asin": meta["parent_asin"],
        "product_name": meta["title"],
        "brand": meta["store"],
        "avg_stars": meta["average_rating"],
        "total_reviews": meta["rating_number"],
        "subcategory": paths.map(subcategory),
        "product_type": paths.map(product_type),
        "category_path": paths.map(" > ".join),
    })

    return products

def load_products(category, wanted_asins):
    path = DATA_DIR / f"products_{category}_{SAMPLE_SIZE}.parquet"

    if path.exists():
        print(f"Loaded{path}")
        return pd.read_parquet(path)

    products = build_products(scan_metadata(category, wanted_asins))
    products.to_parquet(path, index=False)

    return products

In [10]:
# Adds product names, categories, average stars and review counts to every review
category_products = []
for c in CATEGORIES:
    wanted_asins = set(reviews.loc[reviews["category"] == c, "parent_asin"])
    category_products.append(load_products(c, wanted_asins))

products = pd.concat(category_products, ignore_index=True).drop_duplicates("parent_asin")

NAMED_COLUMNS = ["category", "subcategory", "product_type", "product_name", "brand", "avg_stars", "total_reviews", "rating", "review_title", "text", "verified_purchase", "helpful_vote", "timestamp", "parent_asin"]

named = reviews.rename(columns={"title": "review_title"}).merge(products, on="parent_asin", how="left")[NAMED_COLUMNS]
named.head()

Loadeddata/products_Electronics_20000.parquet
Loadeddata/products_Home_and_Kitchen_20000.parquet


,category,subcategory,product_type,product_name,brand,avg_stars,total_reviews,rating,review_title,text,verified_purchase,helpful_vote,timestamp,parent_asin
0,Electronics,Home Audio,Connectors & Adapters,Grado Mini Adaptor,Grado,4.5,339,5.0,Good for the application,Works very well. Plugging on either end is ni...,False,0,2014-09-28 01:01:34.000,B00WTZGRRE
1,Electronics,Camera & Photo,Complete Tripods,"Mini Phone Tripod, ZUMA Premium Mini Tripod fo...",ZUMA PHOTO.VIDEO,4.0,166,5.0,Stand,Does just what it says,True,0,2021-12-22 03:30:12.537,B08RYZ219J
2,Electronics,Computers & Accessories,USB Cables,Maeline Micro USB Cable USB 2.0 A-Male to Micr...,Maeline,4.6,142,5.0,Maeline Micro USB charging cable,"Best cables I've ever had fast charging, stron...",True,0,2022-06-21 18:22:09.417,B07Q3BBNS8
3,Electronics,Computers & Accessories,Screen Protectors,ArmorSuit MilitaryShield Screen Protector for ...,ArmorSuit,4.1,162,5.0,Easy,Works great. I had to reapply it a second time...,True,1,2016-01-28 17:06:50.000,B015JX3H48
4,Electronics,Computers & Accessories,USB-to-VGA Adapters,TWOPAN USB C Hub to VGA Adapter Type-C Hub wit...,TWOPAN,4.1,183,5.0,Nice and Sleek and it works,This is working great for my work MacBook to c...,False,0,2019-12-21 05:07:25.105,B0B28GXPRR


In [11]:
# Checks how many reviews matched a product and lists the most common product types
print(f"{named['product_name'].notna().mean():.1%} of reviews matched a product name")
print(f"{named['product_type'].notna().mean():.1%} have a category path")

named.groupby(["category", "product_type"]).size().sort_values(ascending=False).head(20)

100.0% of reviews matched a product name
93.5% have a category path


category          product_type               
Electronics       Cases                          989
                  Earbud Headphones              810
                  USB Cables                     604
                  Portable Bluetooth Speakers    421
                  Power Strips                   371
                  Lightning Cables               349
Home_and_Kitchen  Container Sets                 337
Electronics       Over-Ear Headphones            323
                  Mice                           317
                  Remote Controls                297
                  Stands                         279
Home_and_Kitchen  Sheet & Pillowcase Sets        269
                  Throws                         248
Electronics       USB Flash Drives               243
Home_and_Kitchen  Throw Pillow Covers            242
Electronics       Micro SD Cards                 203
                  HDMI Cables                    201
                  Screen Protectors              200


In [12]:
MIN_REVIEWS = 1000
MIN_AVG_RATING = 4.0
CANDIDATES_PER_CATEGORY = 150
CANDIDATE_MIN_RATINGS = 5000

LOCAL_REVIEW_FILES = {"Electronics": Path("../project/Electronics.jsonl")}

AVG_RATING_PATTERN = re.compile(rb'"average_rating": ([\d.]+)')
RATING_COUNT_PATTERN = re.compile(rb'"rating_number": (\d+)')


def open_lines(relative_path, local_path=None):
    if local_path is not None and local_path.exists():
        return open(local_path, "rb", buffering=16 * 2**20)

    return io.BufferedReader(urllib.request.urlopen(f"{HF_URL}/{relative_path}"), buffer_size=16 * 2**20)


def find_candidates(category, n=CANDIDATES_PER_CATEGORY):
    rows = []
    with open_lines(f"meta_categories/meta_{category}.jsonl") as lines:
        for line in lines:
            avg = AVG_RATING_PATTERN.search(line)
            count = RATING_COUNT_PATTERN.search(line)

            if avg and count and int(count.group(1)) >= CANDIDATE_MIN_RATINGS and float(avg.group(1)) >= MIN_AVG_RATING:
                item = json.loads(line)
                rows.append(pick_fields(item, META_COLUMNS))

                if len(rows) == n:
                    break

    return build_products(pd.DataFrame(rows, columns=META_COLUMNS))


def collect_reviews(category, wanted_asins):
    rows = []
    local_path = LOCAL_REVIEW_FILES.get(category)
    with open_lines(f"review_categories/{category}.jsonl", local_path) as lines:
        for line_number, line in enumerate(lines, start=1):
            match = PARENT_ASIN_PATTERN.search(line)

            if match and match.group(1).decode() in wanted_asins:
                item = json.loads(line)
                rows.append(pick_fields(item, REVIEW_COLUMNS))

            if line_number % 5_000_000 == 0:
                print(f"  {line_number:,} reviews read, {len(rows):,} kept")

    df = pd.DataFrame(rows, columns=REVIEW_COLUMNS)
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")

    return df


def load_top_products(category, n_products):
    path = DATA_DIR / f"top_reviews_{category.lower()}_{n_products}.parquet"

    if path.exists():
        print(f"{category}: loading saved top products from {path}")
        
        return pd.read_parquet(path)

    print(f"{category}: finding candidates...")
    candidates = find_candidates(category)
    print(f"{category}: reading every review for {len(candidates)} candidates...")
    found = collect_reviews(category, set(candidates["parent_asin"]))

    stats = found.groupby("parent_asin")["rating"].agg(n_reviews="size", avg_rating="mean")
    qualified = stats[(stats["n_reviews"] >= MIN_REVIEWS) & (stats["avg_rating"] >= MIN_AVG_RATING)]

    if len(qualified) < n_products:
        print(f"{category}: only {len(qualified)} candidates qualify (wanted {n_products})")

    chosen = qualified.sample(min(n_products, len(qualified)), random_state=SEED).index

    top = found[found["parent_asin"].isin(chosen)].merge(candidates, on="parent_asin")
    top = top.rename(columns={"title": "review_title"})
    top["category"] = category
    top = top[NAMED_COLUMNS]
    top.to_parquet(path, index=False)
    print(f"{category}: saved {top['parent_asin'].nunique()} products, {len(top):,} reviews to {path}")
    
    return top

In [15]:
# Loads the reviews for the 20 top products and builds a summary table with one row per product
category_tops = []
for c in CATEGORIES:
    category_tops.append(load_top_products(c, TOP_N_TOTAL // len(CATEGORIES)))

top_reviews = pd.concat(category_tops, ignore_index=True)

top_products = (
    top_reviews.assign(positive=top_reviews["rating"] >= 4, negative=top_reviews["rating"] <= 2)
    .groupby(["category", "product_type", "product_name", "brand", "parent_asin"], dropna=False)
    .agg(n_reviews=("rating", "size"), avg_rating=("rating", "mean"),
         n_positive=("positive", "sum"), n_negative=("negative", "sum"))
    .round({"avg_rating": 2})
    .reset_index()
    .sort_values(["category", "n_positive"], ascending=[True, False], ignore_index=True)
)

top_products.head(50)

Electronics: loading saved top products from data/top_reviews_electronics.parquet
Home_and_Kitchen: loading saved top products from data/top_reviews_home_and_kitchen.parquet


,category,product_type,product_name,brand,parent_asin,n_reviews,avg_rating,n_positive,n_negative
0,Electronics,NaN,"Fire HD 8 Tablet (8"" HD Display, 16 GB) - Blac...",Amazon,B07F4P3JH7,30974,4.08,23173,5063
1,Electronics,Cat 6 Cables,"Cat6 Ethernet Cable, 12 Feet (2 Pack) LAN, utp...",Ultra Clarity Cables,B07FTSFGXH,3806,4.72,3564,174
2,Electronics,Backpacks,Tzowla Travel Laptop Backpack Water Resistant ...,Tzowla,B0C49KQRX4,4032,4.49,3511,317
3,Electronics,Monitors,"SAMSUNG 23.5” CF396 Curved Computer Monitor, A...",SAMSUNG,B09SJ9192T,2645,4.10,1992,472
4,Electronics,Messenger & Shoulder Bags,CoolBELL Convertible Backpack Messenger Bag Sh...,CoolBELL,B0B7DTZ18B,1450,4.57,1291,105
5,Electronics,Home Security Systems,"GE Personal Security Window and Door Alarm, 1 ...",GE home electrical,B0BT1TM3S2,1599,4.24,1267,211
6,Electronics,Power Strips,"Anker Power Strip Surge Protector, 12 Outlets ...",Anker,B08Y8FSTMT,1313,4.56,1171,103
7,Electronics,Case Fans,SilverStone Technology SST-FF121 120mm Ultra F...,SilverStone Technology,B07RWDZ3HG,1237,4.46,1071,84
8,Electronics,Batteries,"BTECH, BaoFeng BL-5 Battery Eliminator for for...",BTECH,B0BP3ZSMH5,1259,4.39,1061,132
9,Electronics,Cases,SUPLIK Kids Case for iPad 5th/6th Generation (...,Suplik,B07LBPWW3D,1163,4.39,977,130


In [14]:
# Keeps only the 4 and 5 star reviews of each top product and picks a random base set of 50 per product

positive_reviews = top_reviews[top_reviews["rating"] >= 4]

base_sets = (
    positive_reviews.groupby("parent_asin", group_keys=False)
    .sample(n=REVIEWS_PER_SUMMARY, random_state=SEED)
    .reset_index(drop=True)
)
base_sets.groupby("product_name")["rating"].agg(n_reviews="size", avg_rating="mean").round(2)

,n_reviews,avg_rating
product_name,,
"Anker Power Strip Surge Protector, 12 Outlets & 3 USB Charging Ports with Flat Plug, 6ft Extension Cord, PowerIQ for iPhone Xs/XS Max/XR/X, Galaxy, for Home, Office, and More (4000 Joules) (Black)",50,4.96
"BTECH, BaoFeng BL-5 Battery Eliminator for for BF-F8HP, UV-5X3, and UV-5R Radios",50,4.86
"Big Joe Fuf XXL Foam Filled Bean Bag Chair with Removable Cover, Midnight Plush, 6ft Giant",50,4.90
"Big Red House Heat-Resistant Oven Mitts - Set of 2 Silicone Kitchen Oven Mitt Gloves, Turquoise",50,4.88
"Brightown Hanging Window Curtain Lights 9.8 Feet Dimmable, Connectable with 300 Led, Remote, 8 Lighting Modes, Timer for Bedroom Wall Party Indoor Outdoor Decor, Blue Lights (Curtain is Not Included)",50,4.82
"Cat6 Ethernet Cable, 12 Feet (2 Pack) LAN, utp Cat 6, RJ45, Network Cord, Patch, Internet Cable - 12 ft - Black",50,4.94
CoolBELL Convertible Backpack Messenger Bag Shoulder bag Laptop Case Handbag Business Briefcase Multi-functional Travel Rucksack Fits 15.6 Inch Laptop For Men/Women (Cancas Dark Grey),50,4.86
"Evershine (6-Pack Heavy Duty Plastic Table Covers Tablecloth (Reusable) (Rectangle 54"" x 108"", Black)",50,4.94
"Fire HD 8 Tablet (8"" HD Display, 16 GB) - Black (Previous Generation - 8th)",50,4.78
